In [1]:
print('a')

a


In [2]:
import pandas

In [3]:
from __future__ import annotations

import logging
import os
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

logger = logging.getLogger(__name__)

CATBOOST_PARAMS = {
    "iterations": 300,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3,
    "random_seed": 42,
    "verbose": 0,
    "thread_count": 1,
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
}


@dataclass
class ScoringResult:
    # CV-метрики на train (всегда доступны)
    cv_roc_auc: float = 0.0
    cv_std: float = 0.0
    cv_folds: list = field(default_factory=list)
    # Test-метрики по скрытым меткам (доступны если передан hidden_labels_path)
    test_roc_auc: float | None = None
    test_gini: float | None = None
    # Общая статистика
    n_features: int = 0
    train_rows: int = 0
    test_rows: int = 0
    scoring_elapsed: float = 0.0
    top_features: dict = field(default_factory=dict)


class ScoringEngine:
    def __init__(
        self,
        id_column: str = "client_id",
        target_column: str = "target",
        hidden_labels_path: str | Path | None = None,
    ):
        self.id_column = id_column
        self.target_column = target_column
        self.hidden_labels_path = Path(hidden_labels_path) if hidden_labels_path else None

    def score(self, output_dir: str) -> ScoringResult:
        t_start = time.perf_counter()

        train_df = pd.read_csv(os.path.join(output_dir, "train.csv"))
        test_df = pd.read_csv(os.path.join(output_dir, "test.csv"))

        feature_cols = [
            c for c in train_df.columns if c not in (self.id_column, self.target_column)
        ]
        logger.info("[scoring] Признаки (%d): %s", len(feature_cols), feature_cols)

        X_train = train_df[feature_cols].copy()
        y_train = train_df[self.target_column].copy()
        X_test = test_df[feature_cols].copy()

        X_train = X_train.fillna(-999)
        X_test = X_test.fillna(-999)

        cat_features = []
        for i, col in enumerate(X_train.columns):
            if X_train[col].dtype == object or str(X_train[col].dtype) == "str":
                X_train[col] = X_train[col].astype(str)
                X_test[col] = X_test[col].astype(str)
                cat_features.append(i)

        # --- Финальная модель на всём train ---
        logger.info("[scoring] Обучение финальной модели...")
        model = CatBoostClassifier(**CATBOOST_PARAMS, cat_features=cat_features or None)
        model.fit(X_train, y_train)

        # --- 5-fold CV на train ---
        logger.info("[scoring] 5-fold CV...")
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = []
        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
            fold_model = CatBoostClassifier(**CATBOOST_PARAMS, cat_features=cat_features or None)
            fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
            probas = fold_model.predict_proba(X_train.iloc[val_idx])[:, 1]
            fold_auc = roc_auc_score(y_train.iloc[val_idx], probas)
            cv_scores.append(fold_auc)
            logger.info("[scoring]   fold %d: AUC=%.4f", fold, fold_auc)
        cv_scores = np.array(cv_scores)

        # --- Test ROC-AUC по скрытым меткам ---
        test_roc_auc = None
        test_gini = None
        if self.hidden_labels_path and self.hidden_labels_path.exists():
            logger.info("[scoring] Загрузка скрытых меток из %s", self.hidden_labels_path)
            labels_df = pd.read_csv(self.hidden_labels_path)
            merged = test_df[[self.id_column]].merge(labels_df, on=self.id_column, how="left")
            hidden_labels = merged[self.target_column].values

            test_probas = model.predict_proba(X_test)[:, 1]
            test_roc_auc = round(float(roc_auc_score(hidden_labels, test_probas)), 6)
            test_gini = round(2 * test_roc_auc - 1, 6)
            logger.info("[scoring] Test ROC-AUC=%.4f, Gini=%.4f", test_roc_auc, test_gini)
        else:
            logger.info("[scoring] Скрытые метки не найдены, test ROC-AUC не вычислен")

        feature_importance = dict(zip(feature_cols, model.get_feature_importance().tolist()))
        top_features = dict(
            sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:20]
        )

        scoring_elapsed = time.perf_counter() - t_start

        result = ScoringResult(
            cv_roc_auc=round(float(cv_scores.mean()), 6),
            cv_std=round(float(cv_scores.std()), 6),
            cv_folds=[round(float(s), 6) for s in cv_scores],
            test_roc_auc=test_roc_auc,
            test_gini=test_gini,
            n_features=len(feature_cols),
            train_rows=len(X_train),
            test_rows=len(X_test),
            scoring_elapsed=round(scoring_elapsed, 2),
            top_features=top_features,
        )

        logger.info(
            "[scoring] Готово за %.1f сек. CV AUC=%.4f±%.4f, Test AUC=%s",
            scoring_elapsed, result.cv_roc_auc, result.cv_std,
            f"{test_roc_auc:.4f}" if test_roc_auc is not None else "N/A",
        )
        return result


In [26]:
tr = pd.read_csv("C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\data\\train.csv")
df = pd.read_csv("C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\data\\client_data.csv")
ts = pd.read_csv("C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\data\\test.csv")

In [27]:
q = pd.merge(tr, df, on='client_id', how='left')
w = pd.merge(ts, df, on='client_id', how='left')

In [28]:
q.to_csv('C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\output\\train.csv', index=False)
w.to_csv('C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\output\\test.csv', index=False)

In [24]:
ROOT = Path('C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\')
HIDDEN_LABELS_PATH = ROOT / "configs" / "test.csv"
sc = ScoringEngine(
        hidden_labels_path=HIDDEN_LABELS_PATH if HIDDEN_LABELS_PATH.exists() else None,
    )

In [22]:
HIDDEN_LABELS_PATH

WindowsPath('C:/Users/ddstu/Desktop/хакатон/FeaturesAgentTemplate/configs/test.csv')

In [29]:
a = sc.score('C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\output\\')
a

ScoringResult(cv_roc_auc=0.692817, cv_std=0.017721, cv_folds=[0.684093, 0.702722, 0.703069, 0.711854, 0.662345], test_roc_auc=0.650265, test_gini=0.30053, n_features=20, train_rows=32950, test_rows=8238, scoring_elapsed=141.25, top_features={'Unnamed: 0': 13.641871762779324, 'campaign': 10.367567777849667, 'age': 8.647941927625654, 'month': 8.280101537672628, 'euribor3m': 8.230394425977376, 'job': 7.174701836395526, 'education': 7.002127986626457, 'housing': 6.460116720555415, 'day_of_week': 5.49011918864394, 'marital': 4.336356229622491, 'cons.conf.idx': 3.7199189514833892, 'contact': 3.7114251324561316, 'default': 3.0261982804494383, 'loan': 2.8439430221761772, 'poutcome': 2.2988344657869386, 'nr.employed': 1.670967257559951, 'emp.var.rate': 1.5656036497019692, 'cons.price.idx': 0.8667585068324264, 'previous': 0.389492033542167, 'pdays': 0.27555930626298597})

In [30]:
result = a

In [31]:
print("=" * 45)
print(f"  CV ROC-AUC (train)  : {result.cv_roc_auc:.4f} ± {result.cv_std:.4f}")
print(f"  CV folds            : {result.cv_folds}")
if result.test_roc_auc is not None:
    print(f"  Test ROC-AUC        : {result.test_roc_auc:.4f}")
    print(f"  Test Gini           : {result.test_gini:.4f}")
else:
    print("  Test ROC-AUC        : N/A (configs/test.csv не найден)")
print(f"  Features            : {list(result.top_features.keys())}")
print("=" * 45)

  CV ROC-AUC (train)  : 0.6928 ± 0.0177
  CV folds            : [0.684093, 0.702722, 0.703069, 0.711854, 0.662345]
  Test ROC-AUC        : 0.6503
  Test Gini           : 0.3005
  Features            : ['Unnamed: 0', 'campaign', 'age', 'month', 'euribor3m', 'job', 'education', 'housing', 'day_of_week', 'marital', 'cons.conf.idx', 'contact', 'default', 'loan', 'poutcome', 'nr.employed', 'emp.var.rate', 'cons.price.idx', 'previous', 'pdays']


In [32]:
df = pd.read_csv("C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\data\\test.csv")

In [34]:
del df['target']
df

,row_id,user_id,product_id
0,1,111,7644
1,2,111,8073
2,3,111,25985
3,4,111,32727
4,5,111,42562
...,...,...,...
144841,144842,206005,46448
144842,144843,206005,47865
144843,144844,206005,48192
144844,144845,206005,49363


In [35]:
df.to_csv('C:\\Users\\ddstu\\Desktop\\хакатон\\FeaturesAgentTemplate\\data\\test.csv', index=False)